# Day 21 — Project: Materials Inventory Aging Report
**Estimated time:** 90 minutes  
**Learning objectives:**
1. Replicate SAP MB52/MMBE-style inventory aging analysis in Python
2. Build multi-bucket aging logic and flag excess stock using sales velocity
3. Produce a plant-level inventory-at-risk summary with visualisation

---
> **SAP Context:** In SAP, MB52 shows unrestricted stock by plant/storage location; MMBE shows the stock overview hierarchy. Aging analysis is typically done by extracting MSEG (material document movements) and comparing BUDAT (posting date) to today. This notebook replicates that logic from a flat extract.


## 0. Setup & Data Load

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

pd.set_option('display.float_format', '{:,.2f}'.format)
plt.style.use('seaborn-v0_8-whitegrid')

from analytics_bootcamp.config import RAW_DATA_DIR as DATA
# Load materials inventory and sales orders
materials = pd.read_csv(DATA / 'materials_inventory.csv', parse_dates=['LAST_MOVEMENT_DATE'])
sales = pd.read_csv(DATA / 'sales_orders.csv', parse_dates=['ERDAT'])

print('Materials shape:', materials.shape)
print('Sales shape:', sales.shape)
materials.head(3)

## 1. Calculate Days Since Last Movement

In [ ]:
# YOUR SOLUTION
# HINT: Use pd.Timestamp.today() or a reference date; handle NaT (no movement ever)
TODAY = pd.Timestamp('2026-04-01')   # use a fixed reference date for reproducibility

# Calculate days since last movement
materials['DAYS_SINCE_MOVEMENT'] = (TODAY - materials['LAST_MOVEMENT_DATE']).dt.days

# If LAST_MOVEMENT_DATE is NaT, treat as very old (e.g., 999 days)
materials['DAYS_SINCE_MOVEMENT'] = materials['DAYS_SINCE_MOVEMENT'].fillna(999).astype(int)

print(materials[['MATNR', 'MAKTX', 'LAST_MOVEMENT_DATE', 'DAYS_SINCE_MOVEMENT']].head(8))
print('\nNull movement dates:', materials['LAST_MOVEMENT_DATE'].isna().sum())

## 2. Build Aging Buckets

In [ ]:
# YOUR SOLUTION
# Buckets: Current (<=30d), Slow (31-90d), Stagnant (91-180d), Dead (>180d or no movement)

def assign_aging_bucket(days):
    if days <= 30:
        return 'Current (<=30d)'
    elif days <= 90:
        return 'Slow (31-90d)'
    elif days <= 180:
        return 'Stagnant (91-180d)'
    else:
        return 'Dead (>180d)'

materials['AGING_BUCKET'] = materials['DAYS_SINCE_MOVEMENT'].apply(assign_aging_bucket)

# Alternatively with pd.cut (more Pythonic):
# bins = [0, 30, 90, 180, 9999]
# labels = ['Current (<=30d)', 'Slow (31-90d)', 'Stagnant (91-180d)', 'Dead (>180d)']
# materials['AGING_BUCKET'] = pd.cut(materials['DAYS_SINCE_MOVEMENT'], bins=bins, labels=labels)

print(materials['AGING_BUCKET'].value_counts())

## 3. Calculate Inventory Value at Risk by Aging Bucket and Plant

In [ ]:
# YOUR SOLUTION
# Inventory value = LABST (stock qty) * STPRS (standard price)
materials['INVENTORY_VALUE'] = materials['LABST'] * materials['STPRS']

# Aggregate by plant and aging bucket
aging_summary = (
    materials.groupby(['WERKS', 'AGING_BUCKET'], observed=True)
    .agg(
        MATERIAL_COUNT=('MATNR', 'count'),
        TOTAL_QTY=('LABST', 'sum'),
        TOTAL_VALUE=('INVENTORY_VALUE', 'sum')
    )
    .reset_index()
    .sort_values(['WERKS', 'AGING_BUCKET'])
)

# Value at risk = anything NOT in 'Current'
at_risk = aging_summary[aging_summary['AGING_BUCKET'] != 'Current (<=30d)']
print('=== Inventory at Risk by Plant and Aging Bucket ===')
print(aging_summary.to_string(index=False))

## 4. Flag Excess Stock (stock > 3 months of sales velocity)

In [ ]:
# YOUR SOLUTION
# Step 1: Calculate monthly sales velocity per material from sales orders
# Use last 12 months of closed orders

sales_12m = sales[sales['STATUS'].isin(['DELIVERED', 'CLOSED'])].copy()
sales_12m = sales_12m[sales_12m['ERDAT'] >= (TODAY - pd.DateOffset(months=12))]

monthly_velocity = (
    sales_12m.groupby('MATNR')['MENGE']
    .sum()
    .div(12)                  # average monthly units sold
    .reset_index()
    .rename(columns={'MENGE': 'AVG_MONTHLY_UNITS'})
)

# Step 2: Merge velocity onto inventory
materials_v = materials.merge(monthly_velocity, on='MATNR', how='left')
materials_v['AVG_MONTHLY_UNITS'] = materials_v['AVG_MONTHLY_UNITS'].fillna(0)

# Step 3: Months of stock on hand; excess if > 3 months
materials_v['MONTHS_ON_HAND'] = np.where(
    materials_v['AVG_MONTHLY_UNITS'] > 0,
    materials_v['LABST'] / materials_v['AVG_MONTHLY_UNITS'],
    np.inf  # no sales = infinite months on hand
)
materials_v['EXCESS_FLAG'] = materials_v['MONTHS_ON_HAND'] > 3

excess_count = materials_v['EXCESS_FLAG'].sum()
excess_value = materials_v.loc[materials_v['EXCESS_FLAG'], 'INVENTORY_VALUE'].sum()
print(f'Excess materials: {excess_count}')
print(f'Excess inventory value: ${excess_value:,.0f}')
materials_v[materials_v['EXCESS_FLAG']].nlargest(10, 'INVENTORY_VALUE')[
    ['MATNR','MAKTX','WERKS','LABST','AVG_MONTHLY_UNITS','MONTHS_ON_HAND','INVENTORY_VALUE','AGING_BUCKET']
]

## 5. Output: Summary Table + Bar Chart

In [ ]:
# Summary table
summary_pivot = aging_summary.pivot_table(
    index='AGING_BUCKET', columns='WERKS',
    values='TOTAL_VALUE', aggfunc='sum', fill_value=0
)
summary_pivot['TOTAL'] = summary_pivot.sum(axis=1)
print('=== Inventory Value ($) by Aging Bucket and Plant ===')
print(summary_pivot.applymap(lambda x: f'${x:,.0f}'))

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Chart 1: Total value by aging bucket
bucket_totals = aging_summary.groupby('AGING_BUCKET', observed=True)['TOTAL_VALUE'].sum().sort_values(ascending=False)
colors = {'Current (<=30d)': '#2ecc71', 'Slow (31-90d)': '#f39c12',
          'Stagnant (91-180d)': '#e67e22', 'Dead (>180d)': '#e74c3c'}
bar_colors = [colors.get(b, '#95a5a6') for b in bucket_totals.index]
bucket_totals.plot(kind='bar', ax=axes[0], color=bar_colors, edgecolor='white')
axes[0].set_title('Inventory Value by Aging Bucket', fontweight='bold')
axes[0].set_ylabel('Value (USD)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
axes[0].tick_params(axis='x', rotation=30)

# Chart 2: Value at risk by plant
plant_atrisk = at_risk.groupby('WERKS')['TOTAL_VALUE'].sum()
plant_atrisk.plot(kind='bar', ax=axes[1], color='#e74c3c', edgecolor='white')
axes[1].set_title('At-Risk Inventory Value by Plant', fontweight='bold')
axes[1].set_ylabel('Value (USD)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('day21_inventory_aging.png', dpi=120, bbox_inches='tight')
plt.show()
print('Chart saved.')

## Daily Prompt
**Answer independently (no peeking):**

You're a data architect presenting to the Supply Chain VP. The analysis shows $850K of inventory has been sitting untouched for over 6 months across plants 1000 and 2000.

1. Write a 3-sentence executive summary of the findings.
2. What additional data from SAP (e.g., MSEG, MBEW, purchase orders) would you pull to strengthen the recommendation to write off or reposition stock?
3. Modify the `assign_aging_bucket` function to add a fifth bucket: "Critical — No Movement Ever" for materials where `LAST_MOVEMENT_DATE` is null.

Write your answer in a new markdown cell below.
